# Session 7, Part B: Fairness audit: does the model treat groups the same?

> **Build a credit-risk model on the German Credit data, then measure it group by group.** Same evaluation tools as the bank model, error rates, but now split by *who the applicant is*.

The outcome is `target = 1`, **flagged a bad credit risk**. You'll see the model is harder on younger applicants. This notebook puts a number on it and asks you to decide what to do.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from a public link, so there is nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/bu7852/session7/data/german_credit.csv"

import pandas as pd

if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload german_credit.csv ...")
        up = files.upload()
        df = pd.read_csv(next(iter(up)))
    except ImportError:
        df = pd.read_csv("data/german_credit.csv")   # local fallback (course folder: data/german_credit.csv)

df.head()

## Step 1: Build the model

> Ask Claude: *"Fit a logistic model predicting `target`, one-hot encoding the categoricals, 70/30 split, and keep track of each test applicant's age."*

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = pd.get_dummies(df.drop(columns='target'), drop_first=True)
y = df['target']
Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
    X, y, df.index, test_size=0.3, random_state=1, stratify=y)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
pred = clf.predict(Xte)
print('Overall accuracy:', round(accuracy_score(yte, pred), 3))

## Step 2: Split the results by age group

Predict first: *do you expect the error rate and the flag rate to be the same for younger and older applicants?*

> Ask Claude: *"For under-25, 25-54 and 55+ applicants, show the model's error rate and the share it flags as bad risk."*

In [ ]:
age = df.loc[idx_te, 'age'].values
groups = {'under 25': age <= 25, '25-54': (age > 25) & (age < 55), '55 and over': age >= 55}

print(f"{'group':>12} | {'n':>4} | {'error rate':>10} | {'flagged bad':>11}")
for name, m in groups.items():
    err = (pred[m] != yte.values[m]).mean()
    flag = pred[m].mean()
    print(f'{name:>12} | {m.sum():>4} | {err:>10.1%} | {flag:>11.1%}')

## Step 3: How far apart, and which fairness?

The model flags a **larger share of younger applicants** as bad risks. Now the hard part: there is more than one way to be "fair", and you usually **can't have them all at once**.

- **Equal flag rates**: flag the same share of each group. Hitting this may mean treating similar individuals differently to meet a group quota.
- **Equal error rates**: make mistakes at the same rate for each group. Reasonable, but it can pull against equal flag rates when the groups' true base rates differ.

> Ask Claude: *"Explain why equal flag rates and equal error rates can't both hold in general when the base rates differ between groups."* Then note which one you think matters more here, and why.

In [ ]:
# Which fairness definition matters more for a loan decision, and why?
#


## Step 4: Decide (this is the point)

You're advising the lender. Write what you would **do** and what you would **tell the regulator**:

- What would you change, drop `age`, adjust the threshold per group, add human review, re-collect data?
- What's the trade-off of your choice? (Remember: fixing one fairness measure can worsen another.)
- Could you explain a single declined application to that person? "The model said so" is not an answer.

In [ ]:
# Your recommendation to the lender + what you'd tell the regulator:
#


## Step 5: Deploy checklist

Before any model like this goes live, four questions, the same four from the slides:

1. **Is it good?** (precision and recall, not just accuracy)
2. **At what cost?** (the threshold, set by the price of each mistake)
3. **Is it fair?** (group error rates, and which fairness definition you chose)
4. **Who is accountable?** (a named person signs off, not "the model")

> Ask Claude: *"Summarise this German Credit model against all four checklist items in one short paragraph each."*

In [ ]:
# Your one-paragraph answer per checklist item:
#


## Before you leave

1. How can a model be accurate overall and still unfair to a group?
2. Name two fairness definitions and why you can't satisfy both.
3. One sentence for a client who says "just remove age and it's fair". (Hint: proxies, other columns can quietly carry the same information as age.)